In [ ]:
# Cell 1
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/cv-bootcamp
!unzip -o -q code.zip
!unzip -o -q "CV-20261011T165413Z-1-001.zip" -d /content/cvzip
!cp -r /content/cvzip/CV/data /content/data
!ls /content/data
!nvidia-smi | head -12

/content/drive/.shortcut-targets-by-id/1li_1NZ-qyh04IG3I4SuqZM0bP353t4Lp/cv-bootcamp
classes.txt  test_images.npy  train_images.npy
labels.csv   train_idx.txt    val_idx.txt
Sun Oct 18 11:24:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  

In [ ]:
!ls code predictions outputs

code:
common.py  __pycache__	task2.py  task4.py	    task5.py	    train.py
model.py   task1.py	task3.py  task4_relabel.py  test_common.py

outputs:
clean_ref_final.pt	 t4_filter_log.csv	    t5_fold0_logits.pt
clean_ref_log.csv	 t4_filter_test_preds.pt    t5_fold1_logits.pt
clean_ref_test_preds.pt  t4_ls_log.csv		    t5_fold2_logits.pt
noisy_ce_final.pt	 t4_ls_test_preds.pt	    t5_fold3_logits.pt
noisy_ce_log.csv	 t4_relabel_log.csv	    t5_fold4_logits.pt
noisy_ce_test_preds.pt	 t4_relabel_test_preds.pt   task1_summary.json
t1_confusion_matrix.csv  t4_sce_log.csv		    task2_summary.json
t3_fold0_logits.pt	 t4_sce_test_preds.pt	    task3_summary.json
t3_fold1_logits.pt	 t5_baseline_log.csv	    task4_relabel_summary.json
t3_fold2_logits.pt	 t5_baseline_test_preds.pt  task4_summary.json
t3_fold3_logits.pt	 t5_filter_log.csv	    task5_summary.json
t3_fold4_logits.pt	 t5_filter_test_preds.pt

predictions:
t1_clean_ref.csv  t2_early_stop.csv    t4_ls.csv       t5_baseline_synth.csv
t1_noisy_ce

In [ ]:
# Cell 3: GPU smoke test (~1 min)
!python code/task1.py --data /content/data --smoke

device: cuda | epochs: 2
noise rate: 0.4400

=== training clean_ref ===
{'epoch': 1, 'train_loss': 6.0276, 'lr': 0.05, 'val_acc_noisy': 0.105, 'val_acc_clean': 0.085, 'sec': 7.6346}
{'epoch': 2, 'train_loss': 6.372, 'lr': 0.0, 'val_acc_noisy': 0.105, 'val_acc_clean': 0.1, 'sec': 0.202}

=== training noisy_ce ===
{'epoch': 1, 'train_loss': 6.6683, 'lr': 0.05, 'val_acc_noisy': 0.085, 'val_acc_clean': 0.11, 'train_acc_correct': 0.0643, 'train_acc_mislabeled': 0.1, 'sec': 0.6354}
{'epoch': 2, 'train_loss': 7.2315, 'lr': 0.0, 'val_acc_noisy': 0.145, 'val_acc_clean': 0.125, 'train_acc_correct': 0.1, 'train_acc_mislabeled': 0.1, 'sec': 0.2492}

 {
  "noise_rate": 0.44,
  "val_acc_noisy_labels__clean_ref_model": 0.10499999672174454,
  "val_acc_noisy_labels__noisy_ce_model": 0.14499999582767487
}
best noisy-val epoch (noisy run): 2


In [ ]:
# Cell 4: real run (~2 x 60 epochs)
!python code/task1.py --data /content/datam

device: cuda | epochs: 60
noise rate: 0.4024

=== training clean_ref ===
{'epoch': 1, 'train_loss': 1.8628, 'lr': 0.0999, 'val_acc_noisy': 0.3384, 'val_acc_clean': 0.4614, 'sec': 19.7974}
{'epoch': 2, 'train_loss': 1.3102, 'lr': 0.0997, 'val_acc_noisy': 0.283, 'val_acc_clean': 0.3798, 'sec': 16.5172}
{'epoch': 3, 'train_loss': 1.0523, 'lr': 0.0994, 'val_acc_noisy': 0.4048, 'val_acc_clean': 0.5734, 'sec': 16.7446}
{'epoch': 4, 'train_loss': 0.8909, 'lr': 0.0989, 'val_acc_noisy': 0.4878, 'val_acc_clean': 0.71, 'sec': 16.9632}
{'epoch': 5, 'train_loss': 0.7589, 'lr': 0.0983, 'val_acc_noisy': 0.5022, 'val_acc_clean': 0.7408, 'sec': 17.1997}
{'epoch': 6, 'train_loss': 0.6602, 'lr': 0.0976, 'val_acc_noisy': 0.4974, 'val_acc_clean': 0.7402, 'sec': 17.3953}
{'epoch': 7, 'train_loss': 0.5893, 'lr': 0.0967, 'val_acc_noisy': 0.4944, 'val_acc_clean': 0.7296, 'sec': 17.6063}
{'epoch': 8, 'train_loss': 0.5447, 'lr': 0.0957, 'val_acc_noisy': 0.5092, 'val_acc_clean': 0.77, 'sec': 17.6005}
{'epoch': 9,

In [ ]:
%%writefile code/task2.py
"""Task 2: pick early-stop epoch by NOISY val accuracy only, export predictions."""
import os, json, torch, pandas as pd, sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from train import write_pred_csv

log = pd.read_csv("outputs/noisy_ce_log.csv")
preds = torch.load("outputs/noisy_ce_test_preds.pt")      # (60, 10000)
ep = int(log.val_acc_noisy.idxmax()) + 1                  # noisy val only
write_pred_csv(preds[ep - 1], "predictions/t2_early_stop.csv")

# memorization start: first epoch where mislabeled train acc exceeds its early baseline by 3 pts
base = log.train_acc_mislabeled.iloc[:30].mean()
mem = int(log.epoch[log.train_acc_mislabeled > base + 0.03].iloc[0])
out = dict(memorization_start_epoch=mem, early_stop_epoch=ep,
           val_acc_noisy_labels_at_early_stop=float(log.val_acc_noisy.iloc[ep - 1]))
json.dump(out, open("outputs/task2_summary.json", "w"), indent=2)
print(out)

Overwriting code/task2.py


In [ ]:
!python code/task2.py
!ls predictions

{'memorization_start_epoch': 1, 'early_stop_epoch': 35, 'val_acc_noisy_labels_at_early_stop': 0.5586000084877014}
t1_clean_ref.csv  t1_noisy_ce.csv  t2_dynamics.csv  t2_early_stop.csv


In [ ]:
p = "code/task2.py"
s = open(p).read()
s = s.replace("base = log.train_acc_mislabeled.iloc[:30].mean()",
              "base = log.train_acc_mislabeled.iloc[9:30].mean()")
s = s.replace("mem = int(log.epoch[log.train_acc_mislabeled > base + 0.03].iloc[0])",
              "late = log[log.epoch > 30]\nmem = int(late.epoch[late.train_acc_mislabeled > base + 0.03].iloc[0])")
open(p, "w").write(s)

966

In [ ]:
!python code/task2.py

{'memorization_start_epoch': 38, 'early_stop_epoch': 35, 'val_acc_noisy_labels_at_early_stop': 0.5586000084877014}


In [ ]:
%%writefile code/task3.py
"""Task 3: 5-fold cross-validated noise scores. Resumable. Clean labels used only to MEASURE AUROC."""
import os, sys, json, argparse
import numpy as np, pandas as pd, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import load_data
from train import train, get_logits

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data")
ap.add_argument("--out", default=".")
ap.add_argument("--epochs", type=int, default=30)
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    args.out, args.epochs = "/tmp/cv_smoke", 2
dev = "cuda" if torch.cuda.is_available() else "cpu"
pred_dir, out_dir = os.path.join(args.out, "predictions"), os.path.join(args.out, "outputs")
os.makedirs(pred_dir, exist_ok=True); os.makedirs(out_dir, exist_ok=True)

d = load_data(args.data, device=dev)
tidx = np.loadtxt(os.path.join(args.data, "train_idx.txt"), dtype=int)
if args.smoke:
    for k in ["x_train", "y_noisy_train", "y_clean_train", "y_synth_train"]: d[k] = d[k][:500]
    for k in ["x_val", "y_noisy_val", "y_clean_val", "y_synth_val"]: d[k] = d[k][:200]
    d["x_test"] = d["x_test"][:200]; tidx = tidx[:500]
N = len(d["x_train"]); assert len(tidx) == N
yn = d["y_noisy_train"]

g = torch.Generator().manual_seed(0)
folds = torch.randperm(N, generator=g).chunk(5)
logits = torch.zeros(N, 10)

for k in range(5):
    fp = os.path.join(out_dir, f"t3_fold{k}_logits.pt")
    held = folds[k]
    if os.path.exists(fp):
        print(f"[skip] fold {k}"); logits[held] = torch.load(fp); continue
    train_pos = torch.cat([folds[j] for j in range(5) if j != k])
    print(f"=== fold {k}: train on {len(train_pos)}, score {len(held)} ===", flush=True)
    r = train(d, yn, epochs=args.epochs, subset=train_pos, val_sets={"noisy": d["y_noisy_val"]}, verbose=False)
    print(r["log"].iloc[-1][["epoch", "train_loss", "val_acc_noisy", "sec"]].to_dict(), flush=True)
    lg = get_logits(r["net"], d["x_train"][held.to(dev)], d).cpu()
    torch.save(lg, fp); logits[held] = lg

# ---- signals (higher = more likely mislabeled)
lp = torch.log_softmax(logits, 1); p = lp.exp(); ync = yn.cpu()
p_noisy = p[torch.arange(N), ync]
signals = {
    "loss": -lp[torch.arange(N), ync],
    "margin": p.max(1).values - p_noisy,
    "disagree": (p.argmax(1) != ync).float() + 1e-3 * (-lp[torch.arange(N), ync]).clamp(max=100),
}
# ---- measurement with clean labels (allowed: own Task 3 check)
from sklearn.metrics import roc_auc_score
wrong = (d["y_clean_train"] != yn).cpu().numpy(); kk = int(wrong.sum()); res = {}
for name, s in signals.items():
    s = s.numpy(); top = np.argsort(-s)[:kk]
    res[name] = dict(auroc=float(roc_auc_score(wrong, s)), precision_at_k=float(wrong[top].mean()))
    print(name, res[name])
best = max(res, key=lambda n: res[n]["auroc"])
print("best signal:", best)
pd.DataFrame({"index": tidx, "score": signals[best].numpy()}).to_csv(os.path.join(pred_dir, "t3_noise_scores.csv"), index=False)
json.dump(dict(method=f"5-fold CV, {args.epochs} epochs, signal={best}", results=res),
          open(os.path.join(out_dir, "task3_summary.json"), "w"), indent=2)

Writing code/task3.py


In [ ]:
!python code/task3.py --data /content/data --smoke

=== fold 0: train on 400, score 100 ===
{'epoch': 2.0, 'train_loss': 6.862166442871094, 'val_acc_noisy': 0.11999999731779099, 'sec': 0.18273305892944336}
=== fold 1: train on 400, score 100 ===
{'epoch': 2.0, 'train_loss': 6.966982860565185, 'val_acc_noisy': 0.08500000089406967, 'sec': 0.17989802360534668}
=== fold 2: train on 400, score 100 ===
{'epoch': 2.0, 'train_loss': 5.166902465820312, 'val_acc_noisy': 0.11999999731779099, 'sec': 0.1779320240020752}
=== fold 3: train on 400, score 100 ===
{'epoch': 2.0, 'train_loss': 6.353845672607422, 'val_acc_noisy': 0.04999999701976776, 'sec': 0.17948508262634277}
=== fold 4: train on 400, score 100 ===
{'epoch': 2.0, 'train_loss': 8.994754638671875, 'val_acc_noisy': 0.125, 'sec': 0.17548561096191406}
loss {'auroc': 0.5713717532467533, 'precision_at_k': 0.5045454545454545}
margin {'auroc': 0.5548944805194805, 'precision_at_k': 0.5}
disagree {'auroc': 0.5686525974025973, 'precision_at_k': 0.5}
best signal: loss


In [ ]:
!python code/task3.py --data /content/data

=== fold 0: train on 36000, score 9000 ===
{'epoch': 30.0, 'train_loss': 1.144956716855367, 'val_acc_noisy': 0.5335999727249146, 'sec': 14.247548580169678}
=== fold 1: train on 36000, score 9000 ===
{'epoch': 30.0, 'train_loss': 1.112878667778439, 'val_acc_noisy': 0.5381999611854553, 'sec': 14.246976613998413}
=== fold 2: train on 36000, score 9000 ===
{'epoch': 30.0, 'train_loss': 1.1590764971839058, 'val_acc_noisy': 0.5432000160217285, 'sec': 14.257883548736572}
=== fold 3: train on 36000, score 9000 ===
{'epoch': 30.0, 'train_loss': 1.2060466486612955, 'val_acc_noisy': 0.5455999970436096, 'sec': 14.247462511062622}
=== fold 4: train on 36000, score 9000 ===
{'epoch': 30.0, 'train_loss': 1.1488889028761122, 'val_acc_noisy': 0.5419999957084656, 'sec': 14.244132280349731}
loss {'auroc': 0.9274432286947543, 'precision_at_k': 0.8166657463139875}
margin {'auroc': 0.8980162908441016, 'precision_at_k': 0.8238444972113314}
disagree {'auroc': 0.9280990218190818, 'precision_at_k': 0.8180462753

In [ ]:
%%writefile code/task4.py
"""Task 4: robust training methods (same recipe as Task 1). Resumable."""
import os, sys, json, argparse
import numpy as np, pandas as pd, torch, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import load_data
from train import train, write_pred_csv

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data")
ap.add_argument("--out", default=".")
ap.add_argument("--epochs", type=int, default=60)
ap.add_argument("--methods", default="ls,sce,filter")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    args.out, args.epochs = "/tmp/cv_smoke", 2
dev = "cuda" if torch.cuda.is_available() else "cpu"
pred_dir, out_dir = os.path.join(args.out, "predictions"), os.path.join(args.out, "outputs")
os.makedirs(pred_dir, exist_ok=True); os.makedirs(out_dir, exist_ok=True)

d = load_data(args.data, device=dev)
if args.smoke:
    for k in ["x_train", "y_noisy_train", "y_clean_train", "y_synth_train"]: d[k] = d[k][:500]
    for k in ["x_val", "y_noisy_val", "y_clean_val", "y_synth_val"]: d[k] = d[k][:200]
    d["x_test"] = d["x_test"][:200]
yn = d["y_noisy_train"]
vs = {"noisy": d["y_noisy_val"]}

def ls_loss(out, y):
    return F.cross_entropy(out, y, label_smoothing=0.2)

def sce_loss(out, y, alpha=0.5, beta=1.0, A=-4.0):
    ce = F.cross_entropy(out, y)
    p = F.softmax(out, 1).gather(1, y[:, None]).squeeze(1)
    rce = -A * (1 - p).mean()          # reverse CE with log(0) clipped to A
    return alpha * ce + beta * rce

def filter_subset(drop=0.35):
    s = pd.read_csv(os.path.join(pred_dir if not args.smoke else "predictions", "t3_noise_scores.csv"))["score"].values
    s = s[:len(yn)]
    keep = np.where(s <= np.quantile(s, 1 - drop))[0]
    return torch.as_tensor(keep, dtype=torch.long)

cfg = {"ls": dict(loss_fn=ls_loss), "sce": dict(loss_fn=sce_loss),
       "filter": dict(subset="FILTER")}
summary = {}
for m in args.methods.split(","):
    lp, pp = os.path.join(out_dir, f"t4_{m}_log.csv"), os.path.join(out_dir, f"t4_{m}_test_preds.pt")
    if os.path.exists(lp) and os.path.exists(pp):
        log, preds = pd.read_csv(lp), torch.load(pp); print(f"[skip] {m}")
    else:
        print(f"\n=== {m} ===", flush=True)
        kw = dict(cfg[m])
        if kw.get("subset") == "FILTER": kw["subset"] = filter_subset()
        r = train(d, yn, epochs=args.epochs, val_sets=vs, verbose=False, **kw)
        log, preds = r["log"], r["test_preds"]
        log.to_csv(lp, index=False); torch.save(preds, pp)
    write_pred_csv(preds[-1], os.path.join(pred_dir, f"t4_{m}.csv"))
    summary[m] = dict(final_val_acc_noisy=float(log.val_acc_noisy.iloc[-1]),
                      best_val_acc_noisy=float(log.val_acc_noisy.max()))
    print(m, summary[m], flush=True)
json.dump(summary, open(os.path.join(out_dir, "task4_summary.json"), "w"), indent=2)

Writing code/task4.py


In [ ]:
!python code/task4.py --data /content/data --smoke


=== ls ===
ls {'final_val_acc_noisy': 0.08500000089406967, 'best_val_acc_noisy': 0.08500000089406967}

=== sce ===
sce {'final_val_acc_noisy': 0.125, 'best_val_acc_noisy': 0.125}

=== filter ===
filter {'final_val_acc_noisy': 0.10499999672174454, 'best_val_acc_noisy': 0.11499999463558197}


In [ ]:
!python code/task4.py --data /content/data


=== ls ===
ls {'final_val_acc_noisy': 0.5091999769210815, 'best_val_acc_noisy': 0.5593999624252319}

=== sce ===
sce {'final_val_acc_noisy': 0.5515999794006348, 'best_val_acc_noisy': 0.5636000037193298}

=== filter ===
filter {'final_val_acc_noisy': 0.5577999949455261, 'best_val_acc_noisy': 0.5604000091552734}


In [ ]:
%%writefile code/task5.py
"""Task 5: synthetic (symmetric) noise. CE baseline vs filter+retrain. Model selection on SYNTH val only."""
import os, sys, json, argparse
import numpy as np, pandas as pd, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import load_data
from train import train, get_logits, write_pred_csv

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data")
ap.add_argument("--out", default=".")
ap.add_argument("--epochs", type=int, default=60)
ap.add_argument("--cv_epochs", type=int, default=30)
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    args.out, args.epochs, args.cv_epochs = "/tmp/cv_smoke", 2, 2
dev = "cuda" if torch.cuda.is_available() else "cpu"
pred_dir, out_dir = os.path.join(args.out, "predictions"), os.path.join(args.out, "outputs")
os.makedirs(pred_dir, exist_ok=True); os.makedirs(out_dir, exist_ok=True)

d = load_data(args.data, device=dev)
if args.smoke:
    for k in ["x_train", "y_noisy_train", "y_clean_train", "y_synth_train"]: d[k] = d[k][:500]
    for k in ["x_val", "y_noisy_val", "y_clean_val", "y_synth_val"]: d[k] = d[k][:200]
    d["x_test"] = d["x_test"][:200]
ys = d["y_synth_train"]; N = len(ys)
vs = {"synth": d["y_synth_val"]}
print("synthetic noise rate:", float((ys != d["y_clean_train"]).float().mean()))   # measurement only

def run(name, **kw):
    lp, pp = os.path.join(out_dir, f"{name}_log.csv"), os.path.join(out_dir, f"{name}_test_preds.pt")
    if os.path.exists(lp) and os.path.exists(pp):
        print(f"[skip] {name}"); return pd.read_csv(lp), torch.load(pp)
    print(f"\n=== {name} ===", flush=True)
    r = train(d, ys, val_sets=vs, verbose=False, **kw)
    r["log"].to_csv(lp, index=False); torch.save(r["test_preds"], pp)
    return r["log"], r["test_preds"]

# 1. CE baseline on synthetic labels
b_log, b_pred = run("t5_baseline", epochs=args.epochs)
write_pred_csv(b_pred[-1], os.path.join(pred_dir, "t5_baseline_synth.csv"))
print("baseline final synth-val:", float(b_log.val_acc_synth.iloc[-1]), flush=True)

# 2. cross-validated scores on synthetic labels (same signal as Task 3: disagreement + loss tie-break)
folds = torch.randperm(N, generator=torch.Generator().manual_seed(0)).chunk(5)
logits = torch.zeros(N, 10)
for k in range(5):
    fp = os.path.join(out_dir, f"t5_fold{k}_logits.pt"); held = folds[k]
    if os.path.exists(fp):
        print(f"[skip] fold {k}"); logits[held] = torch.load(fp); continue
    tr = torch.cat([folds[j] for j in range(5) if j != k])
    print(f"=== synth CV fold {k} ===", flush=True)
    r = train(d, ys, epochs=args.cv_epochs, subset=tr, val_sets=vs, verbose=False)
    lg = get_logits(r["net"], d["x_train"][held.to(dev)], d).cpu()
    torch.save(lg, fp); logits[held] = lg
lp_ = torch.log_softmax(logits, 1); yc = ys.cpu()
loss = -lp_[torch.arange(N), yc]
score = (lp_.argmax(1) != yc).float() + 1e-3 * loss.clamp(max=100)
drop = 0.35
keep = torch.as_tensor(np.where(score.numpy() <= np.quantile(score.numpy(), 1 - drop))[0], dtype=torch.long)
wrong = (ys != d["y_clean_train"]).cpu().numpy()                                  # measurement only
print(f"filter keeps {len(keep)}; fraction of dropped that are truly wrong: {wrong[np.setdiff1d(np.arange(N), keep.numpy())].mean():.3f}")

# 3. retrain on kept images
f_log, f_pred = run("t5_filter", epochs=args.epochs, subset=keep)
write_pred_csv(f_pred[-1], os.path.join(pred_dir, "t5_best_synth.csv"))
summary = dict(baseline_final_val_acc_synth=float(b_log.val_acc_synth.iloc[-1]),
               filter_final_val_acc_synth=float(f_log.val_acc_synth.iloc[-1]))
json.dump(summary, open(os.path.join(out_dir, "task5_summary.json"), "w"), indent=2)
print(summary)

Writing code/task5.py


In [ ]:
!python code/task5.py --data /content/data --smoke

synthetic noise rate: 0.43800002336502075

=== t5_baseline ===
baseline final synth-val: 0.10499999672174454
=== synth CV fold 0 ===
=== synth CV fold 1 ===
=== synth CV fold 2 ===
=== synth CV fold 3 ===
=== synth CV fold 4 ===
/content/drive/MyDrive/cv-bootcamp/code/task5.py:62: RuntimeWarning: Mean of empty slice.
  print(f"filter keeps {len(keep)}; fraction of dropped that are truly wrong: {wrong[np.setdiff1d(np.arange(N), keep.numpy())].mean():.3f}")
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
filter keeps 500; fraction of dropped that are truly wrong: nan

=== t5_filter ===
{'baseline_final_val_acc_synth': 0.10499999672174454, 'filter_final_val_acc_synth': 0.10499999672174454}


In [ ]:
!python code/task5.py --data /content/data --smoke

synthetic noise rate: 0.43800002336502075
[skip] t5_baseline
baseline final synth-val: 0.1049999967217445
[skip] fold 0
[skip] fold 1
[skip] fold 2
[skip] fold 3
[skip] fold 4
/content/drive/MyDrive/cv-bootcamp/code/task5.py:62: RuntimeWarning: Mean of empty slice.
  print(f"filter keeps {len(keep)}; fraction of dropped that are truly wrong: {wrong[np.setdiff1d(np.arange(N), keep.numpy())].mean():.3f}")
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
filter keeps 500; fraction of dropped that are truly wrong: nan
[skip] t5_filter
{'baseline_final_val_acc_synth': 0.1049999967217445, 'filter_final_val_acc_synth': 0.1049999967217445}


In [ ]:
!python code/task5.py --data /content/data

synthetic noise rate: 0.4009777903556824

=== t5_baseline ===
baseline final synth-val: 0.4244000017642975
=== synth CV fold 0 ===
=== synth CV fold 1 ===
=== synth CV fold 2 ===
=== synth CV fold 3 ===
=== synth CV fold 4 ===
filter keeps 29250; fraction of dropped that are truly wrong: 0.923

=== t5_filter ===
{'baseline_final_val_acc_synth': 0.4244000017642975, 'filter_final_val_acc_synth': 0.4997999966144562}


In [ ]:
%%writefile code/task4_relabel.py
"""Extra Task 4 candidate: filter (top 20%) + relabel (next 15%) with CV predictions, then retrain. Same recipe."""
import os, sys, json, argparse
import numpy as np, pandas as pd, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import load_data
from train import train, write_pred_csv

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data")
ap.add_argument("--out", default=".")
ap.add_argument("--epochs", type=int, default=60)
ap.add_argument("--drop", type=float, default=0.20)
ap.add_argument("--relabel", type=float, default=0.15)
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    args.out, args.epochs = "/tmp/cv_smoke", 2
dev = "cuda" if torch.cuda.is_available() else "cpu"
pred_dir, out_dir = os.path.join(args.out, "predictions"), os.path.join(args.out, "outputs")

d = load_data(args.data, device=dev)
if args.smoke:
    for k in ["x_train", "y_noisy_train", "y_clean_train", "y_synth_train"]: d[k] = d[k][:500]
    for k in ["x_val", "y_noisy_val", "y_clean_val", "y_synth_val"]: d[k] = d[k][:200]
    d["x_test"] = d["x_test"][:200]
yn = d["y_noisy_train"].cpu(); N = len(yn)

# rebuild the Task 3 cross-validated logits (same fold split as task3.py)
folds = torch.randperm(N, generator=torch.Generator().manual_seed(0)).chunk(5)
logits = torch.zeros(N, 10)
for k in range(5):
    logits[folds[k]] = torch.load(os.path.join(out_dir, f"t3_fold{k}_logits.pt"))
lp = torch.log_softmax(logits, 1); pred = lp.argmax(1)
score = (pred != yn).float() + 1e-3 * (-lp[torch.arange(N), yn]).clamp(max=100)   # same signal as Task 3

order = torch.argsort(score, descending=True)
n_drop, n_rel = int(args.drop * N), int(args.relabel * N)
drop_idx, rel_idx = order[:n_drop], order[n_drop:n_drop + n_rel]
keep = torch.ones(N, dtype=torch.bool); keep[drop_idx] = False
new_labels = yn.clone(); new_labels[rel_idx] = pred[rel_idx]

# measurement only (clean labels never used for training/selection)
yc = d["y_clean_train"].cpu()
print(f"dropped {n_drop}: {(yc[drop_idx] != yn[drop_idx]).float().mean():.3f} truly wrong")
print(f"relabeled {n_rel}: old label correct {(yc[rel_idx] == yn[rel_idx]).float().mean():.3f} -> new label correct {(yc[rel_idx] == new_labels[rel_idx]).float().mean():.3f}", flush=True)

r = train(d, new_labels.to(dev), epochs=args.epochs, subset=torch.where(keep)[0],
          val_sets={"noisy": d["y_noisy_val"], "clean": d["y_clean_val"]}, verbose=False)
log, preds = r["log"], r["test_preds"]
os.makedirs(out_dir, exist_ok=True); os.makedirs(pred_dir, exist_ok=True)
log.to_csv(os.path.join(out_dir, "t4_relabel_log.csv"), index=False)
torch.save(preds, os.path.join(out_dir, "t4_relabel_test_preds.pt"))
write_pred_csv(preds[-1], os.path.join(pred_dir, "t4_relabel.csv"))
s = dict(final_val_acc_noisy=float(log.val_acc_noisy.iloc[-1]), best_val_acc_noisy=float(log.val_acc_noisy.max()),
         final_val_acc_clean=float(log.val_acc_clean.iloc[-1]))
json.dump(s, open(os.path.join(out_dir, "task4_relabel_summary.json"), "w"), indent=2)
print(s)

Writing code/task4_relabel.py


In [ ]:
!python code/task4_relabel.py --data /content/data --smoke

dropped 100: 0.460 truly wrong
relabeled 75: old label correct 0.427 -> new label correct 0.080
{'final_val_acc_noisy': 0.11999999731779099, 'best_val_acc_noisy': 0.11999999731779099, 'final_val_acc_clean': 0.09999999403953552}


In [ ]:
!python code/task4_relabel.py --data /content/data

dropped 9000: 0.946 truly wrong
relabeled 6750: old label correct 0.279 -> new label correct 0.504
{'final_val_acc_noisy': 0.548799991607666, 'best_val_acc_noisy': 0.550599992275238, 'final_val_acc_clean': 0.7797999978065491}


In [ ]:
p = "code/train.py"
s = open(p).read()
if "stop_epoch" not in s:
    s = s.replace("seed=SEED, verbose=True):", "seed=SEED, verbose=True, stop_epoch=None):")
    s = s.replace("for ep in range(1, epochs + 1):", "for ep in range(1, (stop_epoch or epochs) + 1):")
    assert "stop_epoch or epochs" in s and "stop_epoch=None" in s
    open(p, "w").write(s)
import glob, os
for f in glob.glob("outputs/t5_fold*_logits.pt") + ["outputs/t5_filter_log.csv", "outputs/t5_filter_test_preds.pt", "outputs/task5_summary.json"]:
    if os.path.exists(f): os.remove(f)
print("patched; baseline kept:", os.path.exists("outputs/t5_baseline_log.csv"))

patched; baseline kept: True


In [ ]:
%%writefile code/task5.py
"""Task 5: synthetic noise. CE baseline vs filter+retrain. Filter method uses 2 networks:
(1) scoring net, 60-epoch schedule stopped at epoch 30 (pre-memorization), scores its own training images;
(2) retrain on kept images. Model selection on SYNTH val only; clean labels for MEASUREMENT only."""
import os, sys, json, argparse
import numpy as np, pandas as pd, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import load_data
from train import train, get_logits, write_pred_csv

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data")
ap.add_argument("--out", default=".")
ap.add_argument("--epochs", type=int, default=60)
ap.add_argument("--score_epoch", type=int, default=30)
ap.add_argument("--drop", type=float, default=0.35)
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    args.out, args.epochs, args.score_epoch = "/tmp/cv_smoke", 2, 1
dev = "cuda" if torch.cuda.is_available() else "cpu"
pred_dir, out_dir = os.path.join(args.out, "predictions"), os.path.join(args.out, "outputs")
os.makedirs(pred_dir, exist_ok=True); os.makedirs(out_dir, exist_ok=True)

d = load_data(args.data, device=dev)
if args.smoke:
    for k in ["x_train", "y_noisy_train", "y_clean_train", "y_synth_train"]: d[k] = d[k][:500]
    for k in ["x_val", "y_noisy_val", "y_clean_val", "y_synth_val"]: d[k] = d[k][:200]
    d["x_test"] = d["x_test"][:200]
ys = d["y_synth_train"]; N = len(ys); vs = {"synth": d["y_synth_val"]}
print("synthetic noise rate:", float((ys != d["y_clean_train"]).float().mean()))

def run(name, **kw):
    lp, pp = os.path.join(out_dir, f"{name}_log.csv"), os.path.join(out_dir, f"{name}_test_preds.pt")
    if os.path.exists(lp) and os.path.exists(pp):
        print(f"[skip] {name}"); return pd.read_csv(lp), torch.load(pp)
    print(f"\n=== {name} ===", flush=True)
    r = train(d, ys, val_sets=vs, verbose=False, **kw)
    r["log"].to_csv(lp, index=False); torch.save(r["test_preds"], pp)
    run.net = r["net"]
    return r["log"], r["test_preds"]

# 1. CE baseline
b_log, b_pred = run("t5_baseline", epochs=args.epochs)
write_pred_csv(b_pred[-1], os.path.join(pred_dir, "t5_baseline_synth.csv"))
print("baseline final synth-val:", float(b_log.val_acc_synth.iloc[-1]), flush=True)

# 2. network 1: scoring net (stopped before memorization), scores its own training images
sp = os.path.join(out_dir, "t5_score_logits.pt")
if os.path.exists(sp):
    logits = torch.load(sp); print("[skip] scoring net")
else:
    print("\n=== scoring net ===", flush=True)
    r = train(d, ys, epochs=args.epochs, stop_epoch=args.score_epoch, val_sets=vs, verbose=False)
    logits = get_logits(r["net"], d["x_train"], d).cpu(); torch.save(logits, sp)
lp_ = torch.log_softmax(logits, 1); yc = ys.cpu()
loss = -lp_[torch.arange(N), yc]
score = (lp_.argmax(1) != yc).float() + 1e-3 * loss.clamp(max=100)
wrong = (ys != d["y_clean_train"]).cpu().numpy()                      # measurement only
from sklearn.metrics import roc_auc_score
print(f"score AUROC (measurement): {roc_auc_score(wrong, score.numpy()):.4f}")
keep_np = np.where(score.numpy() <= np.quantile(score.numpy(), 1 - args.drop))[0]
keep = torch.as_tensor(keep_np, dtype=torch.long)
dropped = np.setdiff1d(np.arange(N), keep_np)
print(f"keeps {len(keep)}; fraction of dropped truly wrong: {wrong[dropped].mean():.3f}")

# 3. network 2: retrain on kept images
f_log, f_pred = run("t5_filter", epochs=args.epochs, subset=keep)
write_pred_csv(f_pred[-1], os.path.join(pred_dir, "t5_best_synth.csv"))
summary = dict(baseline_final_val_acc_synth=float(b_log.val_acc_synth.iloc[-1]),
               filter_final_val_acc_synth=float(f_log.val_acc_synth.iloc[-1]),
               networks_for_filter_method=2)
json.dump(summary, open(os.path.join(out_dir, "task5_summary.json"), "w"), indent=2)
print(summary)

Overwriting code/task5.py


In [ ]:
!python code/task5.py --data /content/data --smoke

synthetic noise rate: 0.43800002336502075

=== t5_baseline ===
baseline final synth-val: 0.10999999940395355

=== scoring net ===
score AUROC (measurement): 0.4933
/content/drive/.shortcut-targets-by-id/1li_1NZ-qyh04IG3I4SuqZM0bP353t4Lp/cv-bootcamp/code/task5.py:64: RuntimeWarning: Mean of empty slice.
  print(f"keeps {len(keep)}; fraction of dropped truly wrong: {wrong[dropped].mean():.3f}")
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
keeps 500; fraction of dropped truly wrong: nan

=== t5_filter ===
{'baseline_final_val_acc_synth': 0.10999999940395355, 'filter_final_val_acc_synth': 0.10999999940395355, 'networks_for_filter_method': 2}


In [ ]:
!python code/task5.py --data /content/data

synthetic noise rate: 0.4009777903556824
[skip] t5_baseline
baseline final synth-val: 0.4244000017642975

=== scoring net ===
score AUROC (measurement): 0.9744
keeps 29251; fraction of dropped truly wrong: 0.951

=== t5_filter ===
{'baseline_final_val_acc_synth': 0.4244000017642975, 'filter_final_val_acc_synth': 0.5041999816894531, 'networks_for_filter_method': 2}


In [ ]:
%%writefile code/task4_coteach.py
"""Task 4 candidate: co-teaching (2 networks, small-loss exchange). Same recipe as Task 1. Forget rate is a fixed hyperparameter."""
import os, sys, json, math, time, argparse
import numpy as np, pandas as pd, torch, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import load_data, prep, set_seed, SEED
from model import ResNet18
from train import get_logits, write_pred_csv, LR, WD, BS, MOMENTUM

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data")
ap.add_argument("--out", default=".")
ap.add_argument("--epochs", type=int, default=60)
ap.add_argument("--forget", type=float, default=0.35)
ap.add_argument("--ramp", type=int, default=10)
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    args.out, args.epochs, args.ramp = "/tmp/cv_smoke", 2, 1
dev = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = dev == "cuda"
pred_dir, out_dir = os.path.join(args.out, "predictions"), os.path.join(args.out, "outputs")
os.makedirs(pred_dir, exist_ok=True); os.makedirs(out_dir, exist_ok=True)

d = load_data(args.data, device=dev)
if args.smoke:
    for k in ["x_train", "y_noisy_train", "y_clean_train", "y_synth_train"]: d[k] = d[k][:500]
    for k in ["x_val", "y_noisy_val", "y_clean_val", "y_synth_val"]: d[k] = d[k][:200]
    d["x_test"] = d["x_test"][:200]
x, yn = d["x_train"], d["y_noisy_train"]; n = len(x); spe = math.ceil(n / BS)

set_seed(SEED)
nets = [ResNet18().to(dev) for _ in range(2)]
opts = [torch.optim.SGD(m.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WD, nesterov=True) for m in nets]
scheds = [torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=args.epochs * spe) for o in opts]
scalers = [torch.amp.GradScaler(dev.type, enabled=use_amp) for _ in nets]

def probs(split):
    return [F.softmax(get_logits(m, d[split], d), 1) for m in nets]

log, test_preds = [], []
for ep in range(1, args.epochs + 1):
    t0 = time.time()
    forget = args.forget * min(1.0, (ep - 1) / args.ramp)
    perm = torch.randperm(n, device=dev); run_loss = 0.0; seen = 0
    for i in range(0, n, BS):
        b = perm[i:i + BS]
        if len(b) < 2: continue
        xb, yb = prep(x[b], d, train=True), yn[b]
        for m in nets: m.train()
        with torch.autocast(device_type=dev, dtype=torch.float16, enabled=use_amp):
            outs = [m(xb) for m in nets]
        losses = [F.cross_entropy(o.float(), yb, reduction="none") for o in outs]
        k = max(1, int(round((1 - forget) * len(b))))
        sel = [torch.argsort(l.detach())[:k] for l in losses]        # small-loss picks of each net
        for j in range(2):                                           # net j learns from the OTHER net's picks
            loss = losses[j][sel[1 - j]].mean()
            opts[j].zero_grad(set_to_none=True)
            scalers[j].scale(loss).backward()
            scalers[j].step(opts[j]); scalers[j].update(); scheds[j].step()
        run_loss += loss.item() * len(b); seen += len(b)
    pv = probs("x_val"); ens = (pv[0] + pv[1]).argmax(1)
    row = {"epoch": ep, "train_loss": run_loss / seen, "forget": forget,
           "val_acc_noisy": (ens == d["y_noisy_val"]).float().mean().item(),
           "val_acc_noisy_net0": (pv[0].argmax(1) == d["y_noisy_val"]).float().mean().item(),
           "val_acc_clean": (ens == d["y_clean_val"]).float().mean().item(),      # measurement only
           "sec": time.time() - t0}
    pt = probs("x_test"); test_preds.append((pt[0] + pt[1]).argmax(1).cpu())
    log.append(row)
    if ep % 10 == 0 or ep == args.epochs:
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()}, flush=True)

log = pd.DataFrame(log); preds = torch.stack(test_preds)
log.to_csv(os.path.join(out_dir, "t4_coteach_log.csv"), index=False)
torch.save(preds, os.path.join(out_dir, "t4_coteach_test_preds.pt"))
write_pred_csv(preds[-1], os.path.join(pred_dir, "t4_coteach.csv"))
s = dict(final_val_acc_noisy=float(log.val_acc_noisy.iloc[-1]), best_val_acc_noisy=float(log.val_acc_noisy.max()),
         final_val_acc_clean=float(log.val_acc_clean.iloc[-1]))
json.dump(s, open(os.path.join(out_dir, "task4_coteach_summary.json"), "w"), indent=2)
print(s)

Overwriting code/task4_coteach.py


In [ ]:
!python code/task4_coteach.py --data /content/data --smoke

{'epoch': 2, 'train_loss': 4.174, 'forget': 0.35, 'val_acc_noisy': 0.1, 'val_acc_noisy_net0': 0.1, 'val_acc_clean': 0.085, 'sec': 0.3793}
{'final_val_acc_noisy': 0.09999999403953552, 'best_val_acc_noisy': 0.14000000059604645, 'final_val_acc_clean': 0.08500000089406967}


In [ ]:
!python code/task4_coteach.py --data /content/data

{'epoch': 10, 'train_loss': 0.7827, 'forget': 0.315, 'val_acc_noisy': 0.4346, 'val_acc_noisy_net0': 0.4172, 'val_acc_clean': 0.5678, 'sec': 34.2111}
{'epoch': 20, 'train_loss': 0.5277, 'forget': 0.35, 'val_acc_noisy': 0.5014, 'val_acc_noisy_net0': 0.4956, 'val_acc_clean': 0.7242, 'sec': 33.7141}
{'epoch': 30, 'train_loss': 0.4288, 'forget': 0.35, 'val_acc_noisy': 0.5262, 'val_acc_noisy_net0': 0.5222, 'val_acc_clean': 0.7502, 'sec': 33.7126}
{'epoch': 40, 'train_loss': 0.3231, 'forget': 0.35, 'val_acc_noisy': 0.555, 'val_acc_noisy_net0': 0.5556, 'val_acc_clean': 0.8088, 'sec': 33.7351}
{'epoch': 50, 'train_loss': 0.1793, 'forget': 0.35, 'val_acc_noisy': 0.5698, 'val_acc_noisy_net0': 0.5682, 'val_acc_clean': 0.8338, 'sec': 33.6593}
{'epoch': 60, 'train_loss': 0.0935, 'forget': 0.35, 'val_acc_noisy': 0.571, 'val_acc_noisy_net0': 0.5706, 'val_acc_clean': 0.832, 'sec': 33.6757}
{'final_val_acc_noisy': 0.5709999799728394, 'best_val_acc_noisy': 0.5735999941825867, 'final_val_acc_clean': 0.831

In [ ]:
p = "code/task4_coteach.py"
s = open(p).read()
s = s.replace('ap.add_argument("--smoke", action="store_true")',
              'ap.add_argument("--smoke", action="store_true")\nap.add_argument("--labels", default="noisy")', 1)
s = s.replace('x, yn = d["x_train"], d["y_noisy_train"]',
              'x = d["x_train"]; yn = d["y_synth_train"] if args.labels == "synth" else d["y_noisy_train"]', 1)
s = s.replace('d["y_noisy_val"]', 'd["y_synth_val" if args.labels == "synth" else "y_noisy_val"]')
s = s.replace('"t4_coteach_log.csv"', '("t5_coteach_log.csv" if args.labels == "synth" else "t4_coteach_log.csv")')
s = s.replace('"t4_coteach_test_preds.pt"', '("t5_coteach_test_preds.pt" if args.labels == "synth" else "t4_coteach_test_preds.pt")')
s = s.replace('"t4_coteach.csv"', '("t5_best_synth.csv" if args.labels == "synth" else "t4_coteach.csv")')
s = s.replace('"task4_coteach_summary.json"', '("task5_coteach_summary.json" if args.labels == "synth" else "task4_coteach_summary.json")')
assert s.count("args.labels") >= 7 and "--labels" in s
open(p, "w").write(s)

4639

In [ ]:
p = "code/task4_coteach.py"
s = open(p).read()
s = s.replace("torch.amp.GradScaler(dev.type, enabled=use_amp)", "torch.amp.GradScaler(dev, enabled=use_amp)")
assert "dev.type" not in s
open(p, "w").write(s)

4224

In [ ]:
!python code/task4_fsce.py --data /content/data --smoke

python3: can't open file '/content/drive/.shortcut-targets-by-id/1li_1NZ-qyh04IG3I4SuqZM0bP353t4Lp/cv-bootcamp/code/task4_fsce.py': [Errno 2] No such file or directory


In [ ]:
%%writefile code/task4_fsce.py
"""Task 4 candidate: filter (Task 3 scores, drop 35%) + symmetric CE loss. Same recipe otherwise."""
import os, sys, json, argparse
import numpy as np, pandas as pd, torch, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import load_data
from train import train, write_pred_csv

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data")
ap.add_argument("--out", default=".")
ap.add_argument("--epochs", type=int, default=60)
ap.add_argument("--drop", type=float, default=0.35)
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    args.out, args.epochs = "/tmp/cv_smoke", 2
dev = "cuda" if torch.cuda.is_available() else "cpu"
pred_dir, out_dir = os.path.join(args.out, "predictions"), os.path.join(args.out, "outputs")
os.makedirs(pred_dir, exist_ok=True); os.makedirs(out_dir, exist_ok=True)

d = load_data(args.data, device=dev)
if args.smoke:
    for k in ["x_train", "y_noisy_train", "y_clean_train", "y_synth_train"]: d[k] = d[k][:500]
    for k in ["x_val", "y_noisy_val", "y_clean_val", "y_synth_val"]: d[k] = d[k][:200]
    d["x_test"] = d["x_test"][:200]
yn = d["y_noisy_train"]; N = len(yn)

def sce_loss(out, y, alpha=0.5, beta=1.0, A=-4.0):
    ce = F.cross_entropy(out, y)
    p = F.softmax(out, 1).gather(1, y[:, None]).squeeze(1)
    return alpha * ce + beta * (-A) * (1 - p).mean()

# Task 3 scores (same fold split / signal as task3.py)
logits = torch.zeros(N, 10)
if args.smoke:
    logits = torch.randn(N, 10)          # smoke test only
else:
    folds = torch.randperm(N, generator=torch.Generator().manual_seed(0)).chunk(5)
    for k in range(5):
        logits[folds[k]] = torch.load(os.path.join(out_dir, f"t3_fold{k}_logits.pt"))
lp = torch.log_softmax(logits, 1); ync = yn.cpu()
score = (lp.argmax(1) != ync).float() + 1e-3 * (-lp[torch.arange(N), ync]).clamp(max=100)
keep = torch.as_tensor(np.where(score.numpy() <= np.quantile(score.numpy(), 1 - args.drop))[0], dtype=torch.long)

r = train(d, yn, epochs=args.epochs, subset=keep, loss_fn=sce_loss,
          val_sets={"noisy": d["y_noisy_val"], "clean": d["y_clean_val"]}, verbose=False)   # clean val = measurement only
log, preds = r["log"], r["test_preds"]
log.to_csv(os.path.join(out_dir, "t4_fsce_log.csv"), index=False)
torch.save(preds, os.path.join(out_dir, "t4_fsce_test_preds.pt"))
write_pred_csv(preds[-1], os.path.join(pred_dir, "t4_fsce.csv"))
s = dict(final_val_acc_noisy=float(log.val_acc_noisy.iloc[-1]), best_val_acc_noisy=float(log.val_acc_noisy.max()),
         final_val_acc_clean=float(log.val_acc_clean.iloc[-1]))
json.dump(s, open(os.path.join(out_dir, "task4_fsce_summary.json"), "w"), indent=2)
print(s)

Writing code/task4_fsce.py


In [ ]:
!python code/task4_fsce.py --data /content/data --smoke

{'final_val_acc_noisy': 0.08500000089406967, 'best_val_acc_noisy': 0.11499999463558197, 'final_val_acc_clean': 0.07000000029802322}


In [ ]:
!python code/task4_fsce.py --data /content/data

{'final_val_acc_noisy': 0.5591999888420105, 'best_val_acc_noisy': 0.5619999766349792, 'final_val_acc_clean': 0.7993999719619751}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/cv-bootcamp
!unzip -o -q "CV-20261011T165413Z-1-001.zip" -d /content/cvzip
!cp -r /content/cvzip/CV/data /content/data
!grep -c '"--labels"' code/task4_coteach.py
!nvidia-smi | head -12

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1li_1NZ-qyh04IG3I4SuqZM0bP353t4Lp/cv-bootcamp
1
Sun Oct 18 18:53:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                          

In [ ]:
!python code/task4_coteach.py --data /content/data --labels synth --smoke

{'epoch': 2, 'train_loss': 2.4299, 'forget': 0.35, 'val_acc_noisy': 0.095, 'val_acc_noisy_net0': 0.115, 'val_acc_clean': 0.05, 'sec': 0.3747}
{'final_val_acc_noisy': 0.0949999988079071, 'best_val_acc_noisy': 0.09999999403953552, 'final_val_acc_clean': 0.04999999701976776}


In [ ]:
!python code/task4_coteach.py --data /content/data --labels synth

{'epoch': 10, 'train_loss': 1.0919, 'forget': 0.315, 'val_acc_noisy': 0.394, 'val_acc_noisy_net0': 0.3912, 'val_acc_clean': 0.6334, 'sec': 34.6749}
{'epoch': 20, 'train_loss': 0.7538, 'forget': 0.35, 'val_acc_noisy': 0.448, 'val_acc_noisy_net0': 0.437, 'val_acc_clean': 0.7484, 'sec': 34.629}
{'epoch': 30, 'train_loss': 0.6369, 'forget': 0.35, 'val_acc_noisy': 0.4692, 'val_acc_noisy_net0': 0.4648, 'val_acc_clean': 0.7768, 'sec': 34.607}
{'epoch': 40, 'train_loss': 0.5042, 'forget': 0.35, 'val_acc_noisy': 0.512, 'val_acc_noisy_net0': 0.4978, 'val_acc_clean': 0.8548, 'sec': 34.5648}
{'epoch': 50, 'train_loss': 0.3128, 'forget': 0.35, 'val_acc_noisy': 0.5202, 'val_acc_noisy_net0': 0.5182, 'val_acc_clean': 0.8758, 'sec': 34.6406}
{'epoch': 60, 'train_loss': 0.1441, 'forget': 0.35, 'val_acc_noisy': 0.5234, 'val_acc_noisy_net0': 0.5154, 'val_acc_clean': 0.877, 'sec': 34.5774}
{'final_val_acc_noisy': 0.5234000086784363, 'best_val_acc_noisy': 0.5238000154495239, 'final_val_acc_clean': 0.8769999

In [ ]:
import json, sys
sys.path.insert(0, "code")
from common import SEED
NAME = "YOUR FULL NAME"
j = lambda p: json.load(open(p))
t1, t2, t3 = (j(f"outputs/task{i}_summary.json") for i in (1, 2, 3))
t4, t4c = j("outputs/task4_summary.json"), j("outputs/task4_coteach_summary.json")
t5, t5c = j("outputs/task5_summary.json"), j("outputs/task5_coteach_summary.json")
m4 = {"sce": t4["sce"]["final_val_acc_noisy"], "filter": t4["filter"]["final_val_acc_noisy"],
      "coteach": t4c["final_val_acc_noisy"]}
res = {
  "name": NAME, "roll_no": "24b0418", "seed": SEED,
  "training_setup": {
    "optimizer": "SGD, momentum 0.9, Nesterov, weight decay 5e-4",
    "learning_rate_schedule": "lr 0.1, cosine decay to 0 per iteration",
    "batch_size": 128, "epochs": 60,
    "augmentation": "random crop with padding + horizontal flip",
    "gpu_used": "Colab T4 (fp16 autocast)"},
  "task1": {k: t1[k] for k in ["noise_rate", "val_acc_noisy_labels__clean_ref_model", "val_acc_noisy_labels__noisy_ce_model"]},
  "task2": {k: t2[k] for k in ["memorization_start_epoch", "early_stop_epoch", "val_acc_noisy_labels_at_early_stop"]},
  "task3": {"method": t3["method"]},
  "task4": {"methods": [{"name": m, "file": f"t4_{m}.csv", "val_acc_noisy_labels": v} for m, v in m4.items()],
            "best_method": max(m4, key=m4.get)},
  "task5": {"val_acc_synthetic_labels__baseline": t5["baseline_final_val_acc_synth"],
            "val_acc_synthetic_labels__best_method": t5c["final_val_acc_noisy"]},
}
json.dump(res, open("results.json", "w"), indent=2)
print(json.dumps(res, indent=2))

{
  "name": "YOUR FULL NAME",
  "roll_no": "24b0418",
  "seed": 0,
  "training_setup": {
    "optimizer": "SGD, momentum 0.9, Nesterov, weight decay 5e-4",
    "learning_rate_schedule": "lr 0.1, cosine decay to 0 per iteration",
    "batch_size": 128,
    "epochs": 60,
    "augmentation": "random crop with padding + horizontal flip",
    "gpu_used": "Colab T4 (fp16 autocast)"
  },
  "task1": {
    "noise_rate": 0.40242222222222224,
    "val_acc_noisy_labels__clean_ref_model": 0.5913999676704407,
    "val_acc_noisy_labels__noisy_ce_model": 0.49639999866485596
  },
  "task2": {
    "memorization_start_epoch": 38,
    "early_stop_epoch": 35,
    "val_acc_noisy_labels_at_early_stop": 0.5586000084877014
  },
  "task3": {
    "method": "5-fold CV, 30 epochs, signal=disagree"
  },
  "task4": {
    "methods": [
      {
        "name": "sce",
        "file": "t4_sce.csv",
        "val_acc_noisy_labels": 0.5515999794006348
      },
      {
        "name": "filter",
        "file": "t4_filter.csv

In [ ]:
%%writefile code/task4_cotplus.py
"""Co-teaching variants: --forget rate, optional --plus (disagreement-based, Co-teaching+). Same recipe as Task 1."""
import os, sys, json, math, time, argparse
import numpy as np, pandas as pd, torch, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import load_data, prep, set_seed, SEED
from model import ResNet18
from train import get_logits, write_pred_csv, LR, WD, BS, MOMENTUM

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data"); ap.add_argument("--out", default=".")
ap.add_argument("--epochs", type=int, default=60)
ap.add_argument("--forget", type=float, default=0.35); ap.add_argument("--ramp", type=int, default=10)
ap.add_argument("--plus", action="store_true"); ap.add_argument("--init", type=int, default=20)
ap.add_argument("--labels", default="noisy"); ap.add_argument("--tag", default="cotplus")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    args.out, args.epochs, args.ramp, args.init = "/tmp/cv_smoke", 2, 1, 1
dev = "cuda" if torch.cuda.is_available() else "cpu"; use_amp = dev == "cuda"
pred_dir, out_dir = os.path.join(args.out, "predictions"), os.path.join(args.out, "outputs")
os.makedirs(pred_dir, exist_ok=True); os.makedirs(out_dir, exist_ok=True)

d = load_data(args.data, device=dev)
if args.smoke:
    for k in ["x_train", "y_noisy_train", "y_clean_train", "y_synth_train"]: d[k] = d[k][:500]
    for k in ["x_val", "y_noisy_val", "y_clean_val", "y_synth_val"]: d[k] = d[k][:200]
    d["x_test"] = d["x_test"][:200]
synth = args.labels == "synth"
x = d["x_train"]; yn = d["y_synth_train"] if synth else d["y_noisy_train"]
yval = d["y_synth_val"] if synth else d["y_noisy_val"]
n = len(x); spe = math.ceil(n / BS)

set_seed(SEED)
nets = [ResNet18().to(dev) for _ in range(2)]
opts = [torch.optim.SGD(m.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WD, nesterov=True) for m in nets]
scheds = [torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=args.epochs * spe) for o in opts]
scalers = [torch.amp.GradScaler(dev, enabled=use_amp) for _ in nets]
probs = lambda split: [F.softmax(get_logits(m, d[split], d), 1) for m in nets]

log, test_preds = [], []
for ep in range(1, args.epochs + 1):
    t0 = time.time()
    forget = args.forget * min(1.0, (ep - 1) / args.ramp)
    perm = torch.randperm(n, device=dev); run_loss = 0.0; seen = 0
    for i in range(0, n, BS):
        b = perm[i:i + BS]
        if len(b) < 2: continue
        xb, yb = prep(x[b], d, train=True), yn[b]
        for m in nets: m.train()
        with torch.autocast(device_type=dev, dtype=torch.float16, enabled=use_amp):
            outs = [m(xb) for m in nets]
        losses = [F.cross_entropy(o.float(), yb, reduction="none") for o in outs]
        pool = torch.arange(len(b), device=dev)
        if args.plus and ep > args.init:
            dis = (outs[0].argmax(1) != outs[1].argmax(1)).nonzero().squeeze(1)
            if len(dis) >= 16: pool = dis                      # disagreement set (else fall back to whole batch)
        k = max(1, int(round((1 - forget) * len(pool))))
        sel = [pool[torch.argsort(l.detach()[pool])[:k]] for l in losses]
        for j in range(2):                                     # net j learns from the OTHER net's picks
            loss = losses[j][sel[1 - j]].mean()
            opts[j].zero_grad(set_to_none=True)
            scalers[j].scale(loss).backward()
            scalers[j].step(opts[j]); scalers[j].update(); scheds[j].step()
        run_loss += loss.item() * len(b); seen += len(b)
    pv = probs("x_val"); ens = (pv[0] + pv[1]).argmax(1)
    row = {"epoch": ep, "train_loss": run_loss / seen, "forget": forget,
           "val_acc_noisy": (ens == yval).float().mean().item(),
           "val_acc_clean": (ens == d["y_clean_val"]).float().mean().item(),   # measurement only
           "sec": time.time() - t0}
    pt = probs("x_test"); test_preds.append((pt[0] + pt[1]).argmax(1).cpu())
    log.append(row)
    if ep % 10 == 0 or ep == args.epochs:
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()}, flush=True)

pre = "t5_" if synth else "t4_"
log = pd.DataFrame(log); preds = torch.stack(test_preds)
log.to_csv(os.path.join(out_dir, f"{pre}{args.tag}_log.csv"), index=False)
torch.save(preds, os.path.join(out_dir, f"{pre}{args.tag}_test_preds.pt"))
write_pred_csv(preds[-1], os.path.join(pred_dir, f"t5_best_synth_{args.tag}.csv" if synth else f"t4_{args.tag}.csv"))
s = dict(final_val_acc_noisy=float(log.val_acc_noisy.iloc[-1]), best_val_acc_noisy=float(log.val_acc_noisy.max()),
         final_val_acc_clean=float(log.val_acc_clean.iloc[-1]))
json.dump(s, open(os.path.join(out_dir, f"{pre}{args.tag}_summary.json"), "w"), indent=2)
print(s)

Overwriting code/task4_cotplus.py


In [ ]:
!python code/task4_cotplus.py --data /content/data --smoke --plus

{'epoch': 2, 'train_loss': 3.6802, 'forget': 0.35, 'val_acc_noisy': 0.095, 'val_acc_clean': 0.115, 'sec': 0.3769}
{'final_val_acc_noisy': 0.0949999988079071, 'best_val_acc_noisy': 0.14000000059604645, 'final_val_acc_clean': 0.11499999463558197}


In [ ]:
!python code/task4_cotplus.py --data /content/data --forget 0.40 --tag f40

{'epoch': 10, 'train_loss': 0.6894, 'forget': 0.36, 'val_acc_noisy': 0.4332, 'val_acc_clean': 0.5716, 'sec': 34.5907}
{'epoch': 20, 'train_loss': 0.4258, 'forget': 0.4, 'val_acc_noisy': 0.5136, 'val_acc_clean': 0.743, 'sec': 34.5083}
{'epoch': 30, 'train_loss': 0.3346, 'forget': 0.4, 'val_acc_noisy': 0.534, 'val_acc_clean': 0.7558, 'sec': 34.4319}
{'epoch': 40, 'train_loss': 0.2377, 'forget': 0.4, 'val_acc_noisy': 0.5446, 'val_acc_clean': 0.7954, 'sec': 34.4296}
{'epoch': 50, 'train_loss': 0.1078, 'forget': 0.4, 'val_acc_noisy': 0.5632, 'val_acc_clean': 0.8274, 'sec': 34.5041}
{'epoch': 60, 'train_loss': 0.0535, 'forget': 0.4, 'val_acc_noisy': 0.5758, 'val_acc_clean': 0.8386, 'sec': 34.3796}
{'final_val_acc_noisy': 0.5758000016212463, 'best_val_acc_noisy': 0.5758000016212463, 'final_val_acc_clean': 0.8385999798774719}


In [ ]:
!python code/task4_cotplus.py --data /content/data --plus --tag plus

{'epoch': 10, 'train_loss': 0.7673, 'forget': 0.315, 'val_acc_noisy': 0.4648, 'val_acc_clean': 0.6166, 'sec': 34.5522}
{'epoch': 20, 'train_loss': 0.5124, 'forget': 0.35, 'val_acc_noisy': 0.5124, 'val_acc_clean': 0.7372, 'sec': 34.498}
{'epoch': 30, 'train_loss': 1.2044, 'forget': 0.35, 'val_acc_noisy': 0.4956, 'val_acc_clean': 0.712, 'sec': 34.7728}
{'epoch': 40, 'train_loss': 0.7804, 'forget': 0.35, 'val_acc_noisy': 0.5312, 'val_acc_clean': 0.762, 'sec': 34.8395}
{'epoch': 50, 'train_loss': 0.2633, 'forget': 0.35, 'val_acc_noisy': 0.5656, 'val_acc_clean': 0.8278, 'sec': 34.7848}
{'epoch': 60, 'train_loss': 0.2041, 'forget': 0.35, 'val_acc_noisy': 0.5712, 'val_acc_clean': 0.8262, 'sec': 34.731}
{'final_val_acc_noisy': 0.5712000131607056, 'best_val_acc_noisy': 0.5712000131607056, 'final_val_acc_clean': 0.826200008392334}


In [4]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/cv-bootcamp
!unzip -o -q "CV-20261011T165413Z-1-001.zip" -d /content/cvzip
!cp -r /content/cvzip/CV/data /content/data
!ls code outputs predictions
!nvidia-smi | head -12

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1li_1NZ-qyh04IG3I4SuqZM0bP353t4Lp/cv-bootcamp
code:
common.py    task1.py  task4_coteach.py  task4.py	   test_common.py
model.py     task2.py  task4_cotplus.py  task4_relabel.py  train.py
__pycache__  task3.py  task4_fsce.py	 task5.py

outputs:
clean_ref_final.pt	  t4_f45_log.csv	     t5_coteach_log.csv
clean_ref_log.csv	  t4_f45_summary.json	     t5_coteach_test_preds.pt
clean_ref_test_preds.pt   t4_f45_test_preds.pt	     t5_f40_log.csv
extras			  t4_filter_log.csv	     t5_f40_summary.json
noisy_ce_final.pt	  t4_filter_test_preds.pt    t5_f40_test_preds.pt
noisy_ce_log.csv	  t4_fsce_log.csv	     t5_filter_log.csv
noisy_ce_test_preds.pt	  t4_fsce_test_preds.pt      t5_filter_test_preds.pt
t1_confusion_matrix.csv   t4_ls_log.csv		     t5_score_logits.pt
t3_fold0_logits.pt	  t4_ls_test_preds.pt	     task1_summary.json
t3_f

In [ ]:
!python code/task4_cotplus.py --data /content/data --forget 0.45 --tag f45

{'epoch': 10, 'train_loss': 0.5991, 'forget': 0.405, 'val_acc_noisy': 0.4536, 'val_acc_clean': 0.6254, 'sec': 33.9583}
{'epoch': 20, 'train_loss': 0.3283, 'forget': 0.45, 'val_acc_noisy': 0.5022, 'val_acc_clean': 0.7262, 'sec': 33.9452}
{'epoch': 30, 'train_loss': 0.2394, 'forget': 0.45, 'val_acc_noisy': 0.5292, 'val_acc_clean': 0.78, 'sec': 33.8699}
{'epoch': 40, 'train_loss': 0.1501, 'forget': 0.45, 'val_acc_noisy': 0.5554, 'val_acc_clean': 0.796, 'sec': 33.842}
{'epoch': 50, 'train_loss': 0.0543, 'forget': 0.45, 'val_acc_noisy': 0.5668, 'val_acc_clean': 0.8416, 'sec': 33.8633}
{'epoch': 60, 'train_loss': 0.0248, 'forget': 0.45, 'val_acc_noisy': 0.5746, 'val_acc_clean': 0.8456, 'sec': 33.8341}
{'final_val_acc_noisy': 0.5745999813079834, 'best_val_acc_noisy': 0.5745999813079834, 'final_val_acc_clean': 0.8455999493598938}


In [ ]:
!python code/task4_cotplus.py --data /content/data --forget 0.40 --tag f40 --labels synth

{'epoch': 10, 'train_loss': 0.9659, 'forget': 0.36, 'val_acc_noisy': 0.3682, 'val_acc_clean': 0.6018, 'sec': 33.3623}
{'epoch': 20, 'train_loss': 0.5879, 'forget': 0.4, 'val_acc_noisy': 0.4504, 'val_acc_clean': 0.7438, 'sec': 33.2466}
{'epoch': 30, 'train_loss': 0.4697, 'forget': 0.4, 'val_acc_noisy': 0.4838, 'val_acc_clean': 0.808, 'sec': 33.2063}
{'epoch': 40, 'train_loss': 0.3328, 'forget': 0.4, 'val_acc_noisy': 0.51, 'val_acc_clean': 0.852, 'sec': 33.2701}
{'epoch': 50, 'train_loss': 0.1781, 'forget': 0.4, 'val_acc_noisy': 0.528, 'val_acc_clean': 0.8864, 'sec': 33.2146}
{'epoch': 60, 'train_loss': 0.0831, 'forget': 0.4, 'val_acc_noisy': 0.53, 'val_acc_clean': 0.8904, 'sec': 33.1561}
{'final_val_acc_noisy': 0.5299999713897705, 'best_val_acc_noisy': 0.5313999652862549, 'final_val_acc_clean': 0.8903999924659729}


In [ ]:
!mkdir -p outputs/extras
!mv predictions/t4_coteach.csv outputs/extras/t4_coteach_f35.csv
!mv predictions/t4_f45.csv predictions/t4_plus.csv outputs/extras/
!mv predictions/t4_f40.csv predictions/t4_coteach.csv
!cp predictions/t5_best_synth.csv outputs/extras/t5_best_synth_coteach_f35.csv
!mv predictions/t5_best_synth_f40.csv predictions/t5_best_synth.csv
!ls predictions

t1_clean_ref.csv     t4_coteach.csv  t4_sce.csv
t1_noisy_ce.csv      t4_filter.csv   t5_baseline_synth.csv
t2_dynamics.csv      t4_fsce.csv     t5_best_synth.csv
t2_early_stop.csv    t4_ls.csv
t3_noise_scores.csv  t4_relabel.csv


In [ ]:
t1_clean_ref.csv     t4_coteach.csv  t4_sce.csv
t1_noisy_ce.csv      t4_filter.csv   t5_baseline_synth.csv
t2_dynamics.csv      t4_fsce.csv     t5_best_synth.csv
t2_early_stop.csv    t4_ls.csv
t3_noise_scores.csv  t4_relabel.csv

SyntaxError: invalid syntax (1452334453.py, line 1)

In [ ]:
!mv predictions/t4_fsce.csv predictions/t4_ls.csv predictions/t4_relabel.csv outputs/extras/
!ls predictions
!echo ---
!ls outputs/extras

t1_clean_ref.csv  t2_early_stop.csv    t4_filter.csv	      t5_best_synth.csv
t1_noisy_ce.csv   t3_noise_scores.csv  t4_sce.csv
t2_dynamics.csv   t4_coteach.csv       t5_baseline_synth.csv
---
t4_coteach_f35.csv  t4_fsce.csv  t4_plus.csv	 t5_best_synth_coteach_f35.csv
t4_f45.csv	    t4_ls.csv	 t4_relabel.csv


In [ ]:
!grep -n -i "pad\|flip\|crop" code/common.py

52:    """x: float (B,3,32,32) in [0,1]. Random horizontal flip + random crop (pad 4)."""
54:    flip = torch.rand(B, device=dev) < 0.5
55:    x = torch.where(flip[:, None, None, None], x.flip(3), x)
56:    xp = F.pad(x, (4, 4, 4, 4))                      # zero padding, like standard CIFAR recipes


In [ ]:
import numpy as np, pandas as pd
d = "/content/data"
lab = pd.read_csv(f"{d}/labels.csv").set_index("index")
tr = np.loadtxt(f"{d}/train_idx.txt", dtype=int)
names = open(f"{d}/classes.txt").read().split()
c, n = lab.loc[tr, "clean_label"].values, lab.loc[tr, "noisy_label"].values
cm = np.zeros((10, 10), int)
np.add.at(cm, (c, n), 1)
print("noise rate", (c != n).mean())
off = cm.copy(); np.fill_diagonal(off, 0)
for f in np.argsort(-off.ravel())[:6]:
    a, b = divmod(f, 10)
    print(f"{names[a]} -> {names[b]}: {off[a, b]} ({off[a, b]/off.sum():.1%} of wrong labels, {off[a, b]/cm[a].sum():.1%} of {names[a]})")

noise rate 0.40242222222222224
truck -> automobile: 1247 (6.9% of wrong labels, 27.9% of truck)
automobile -> truck: 1126 (6.2% of wrong labels, 25.0% of automobile)
cat -> dog: 924 (5.1% of wrong labels, 20.7% of cat)
dog -> cat: 834 (4.6% of wrong labels, 18.4% of dog)
deer -> horse: 683 (3.8% of wrong labels, 15.1% of deer)
deer -> dog: 491 (2.7% of wrong labels, 10.9% of deer)


In [5]:
import pandas as pd
print(pd.read_csv("predictions/t2_dynamics.csv").round(3).to_string(index=False))

 epoch  train_acc_correct  train_acc_mislabeled  val_acc_noisy  val_acc_clean
     1              0.302                 0.169          0.256          0.283
     2              0.417                 0.157          0.314          0.389
     3              0.494                 0.156          0.361          0.456
     4              0.543                 0.144          0.380          0.501
     5              0.593                 0.135          0.396          0.540
     6              0.620                 0.145          0.426          0.580
     7              0.632                 0.130          0.428          0.586
     8              0.681                 0.123          0.450          0.630
     9              0.680                 0.132          0.457          0.626
    10              0.700                 0.144          0.475          0.636
    11              0.679                 0.130          0.447          0.616
    12              0.724                 0.134          0.480  

In [6]:
from google.colab import drive
drive.mount('/content/drive')
!find /content/drive -maxdepth 4 \( -iname "CV-2026*.zip" -o -iname "*bootcamp*" -o -iname "predictions" -o -iname "outputs" -o -name "check_format.py" \) 2>/dev/null

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1li_1NZ-qyh04IG3I4SuqZM0bP353t4Lp/cv-bootcamp
/content/drive/.shortcut-targets-by-id/1li_1NZ-qyh04IG3I4SuqZM0bP353t4Lp/cv-bootcamp/predictions
/content/drive/.shortcut-targets-by-id/1li_1NZ-qyh04IG3I4SuqZM0bP353t4Lp/cv-bootcamp/CV-20261011T165413Z-1-001.zip
/content/drive/.shortcut-targets-by-id/1li_1NZ-qyh04IG3I4SuqZM0bP353t4Lp/cv-bootcamp/outputs
/content/drive/MyDrive/cv-bootcamp
/content/drive/.Encrypted/.shortcut-targets-by-id/1li_1NZ-qyh04IG3I4SuqZM0bP353t4Lp/cv-bootcamp


In [7]:
D = "/content/drive/.shortcut-targets-by-id/1li_1NZ-qyh04IG3I4SuqZM0bP353t4Lp/cv-bootcamp"
!ls -la "{D}"
!ls "{D}/predictions" | wc -l
!ls "{D}/predictions"
!ls "{D}/code" 2>&1 | head -30

total 166240
drwx------ 5 root root      4096 Oct 19 17:26 .
dr-x------ 2 root root      4096 Oct 19 17:26 ..
drwx------ 3 root root      4096 Oct 17 13:39 code
-rw------- 1 root root      7138 Oct 17 13:41 code.zip
-rw------- 1 root root 170095106 Oct 11 16:55 CV-20261011T165413Z-1-001.zip
-rw------- 1 root root    104473 Oct 18 20:42 CVV.ipynb
drwx------ 3 root root      4096 Oct 17 13:52 outputs
drwx------ 2 root root      4096 Oct 17 13:52 predictions
-rw------- 1 root root      1376 Oct 18 19:51 results.json
10
t1_clean_ref.csv  t2_early_stop.csv    t4_filter.csv	      t5_best_synth.csv
t1_noisy_ce.csv   t3_noise_scores.csv  t4_sce.csv
t2_dynamics.csv   t4_coteach.csv       t5_baseline_synth.csv
common.py
model.py
__pycache__
task1.py
task2.py
task3.py
task4_coteach.py
task4_cotplus.py
task4_fsce.py
task4.py
task4_relabel.py
task5.py
test_common.py
train.py


In [8]:
D = "/content/drive/.shortcut-targets-by-id/1li_1NZ-qyh04IG3I4SuqZM0bP353t4Lp/cv-bootcamp"
!mkdir -p /content/cvzip /content/data
!unzip -o -q "{D}/CV-20261011T165413Z-1-001.zip" -d /content/cvzip
!cp -r /content/cvzip/CV/data/* /content/data/
!cp -r "{D}/code" "{D}/outputs" "{D}/predictions" "{D}/results.json" /content/
!find /content/cvzip -name "check_format.py" -o -name "results_template.json"
!ls -R /content/outputs | head -60
!cat /content/results.json
!cat /content/outputs/*summary*.json 2>/dev/null
!cat /content/outputs/*/*summary*.json 2>/dev/null

/content/cvzip/CV/results_template.json
/content/cvzip/CV/check_format.py
/content/outputs:
clean_ref_final.pt
clean_ref_log.csv
clean_ref_test_preds.pt
extras
noisy_ce_final.pt
noisy_ce_log.csv
noisy_ce_test_preds.pt
t1_confusion_matrix.csv
t3_fold0_logits.pt
t3_fold1_logits.pt
t3_fold2_logits.pt
t3_fold3_logits.pt
t3_fold4_logits.pt
t4_coteach_log.csv
t4_coteach_test_preds.pt
t4_f40_log.csv
t4_f40_summary.json
t4_f40_test_preds.pt
t4_f45_log.csv
t4_f45_summary.json
t4_f45_test_preds.pt
t4_filter_log.csv
t4_filter_test_preds.pt
t4_fsce_log.csv
t4_fsce_test_preds.pt
t4_ls_log.csv
t4_ls_test_preds.pt
t4_plus_log.csv
t4_plus_summary.json
t4_plus_test_preds.pt
t4_relabel_log.csv
t4_relabel_test_preds.pt
t4_sce_log.csv
t4_sce_test_preds.pt
t5_baseline_log.csv
t5_baseline_test_preds.pt
t5_coteach_log.csv
t5_coteach_test_preds.pt
t5_f40_log.csv
t5_f40_summary.json
t5_f40_test_preds.pt
t5_filter_log.csv
t5_filter_test_preds.pt
t5_score_logits.pt
task1_summary.json
task2_summary.json
task3_sum

In [9]:
import numpy as np, pandas as pd
d="/content/data"
lab=pd.read_csv(f"{d}/labels.csv").set_index("index")
names=open(f"{d}/classes.txt").read().split()
s=pd.read_csv("/content/predictions/t3_noise_scores.csv"); print(s.head(3))
s=s.set_index(s.columns[0]); sc=s.columns[-1]
y=lab.loc[s.index]; isn=(y.clean_label!=y.noisy_label).values
k=int(isn.sum()); order=s[sc].values.argsort()[::-1]; top=np.zeros(len(s),bool); top[order[:k]]=True
print("prec@k", isn[top].mean())
fp=y[top & ~isn]; print(len(fp),"clean images in top-k; by true class:")
print(fp.clean_label.map(lambda i:names[i]).value_counts().head(4))
ms=y[~top & isn]; print(len(ms),"noisy images missed; by flip:")
print((ms.clean_label.map(lambda i:names[i])+"->"+ms.noisy_label.map(lambda i:names[i])).value_counts().head(4))

   index     score
0      0  1.004346
1      1  0.000514
2      2  1.001546
prec@k 0.8181014964934563
3294 clean images in top-k; by true class:
clean_label
cat     511
bird    467
deer    442
dog     406
Name: count, dtype: int64
3294 noisy images missed; by flip:
truck->automobile    651
cat->dog             444
automobile->truck    382
dog->cat             328
Name: count, dtype: int64


In [10]:
!grep -n -i -E "crop|flip|augment|cutout|erasing|jitter" /content/code/common.py | head
!grep -n "add_argument" /content/code/task*.py
!cat /content/cvzip/CV/results_template.json
!ls -l --time-style=long-iso "{D}/predictions/t5_best_synth.csv" {D}/outputs/t5*summary.json {D}/outputs/task5*summary.json

1:"""Shared utilities: seeding, data loading (GPU-resident), augmentation, evaluation."""
51:def augment(x):
52:    """x: float (B,3,32,32) in [0,1]. Random horizontal flip + random crop (pad 4)."""
54:    flip = torch.rand(B, device=dev) < 0.5
55:    x = torch.where(flip[:, None, None, None], x.flip(3), x)
69:        x = augment(x)
75:    """Eval-mode logits for a uint8 image tensor (no augmentation)."""
/content/code/task1.py:9:ap.add_argument("--data", default="data")
/content/code/task1.py:10:ap.add_argument("--out", default=".", help="project root; predictions/ and outputs/ go here")
/content/code/task1.py:11:ap.add_argument("--epochs", type=int, default=60)
/content/code/task1.py:12:ap.add_argument("--smoke", action="store_true", help="tiny CPU/GPU sanity run, writes to /tmp")
/content/code/task3.py:9:ap.add_argument("--data", default="data")
/content/code/task3.py:10:ap.add_argument("--out", default=".")
/content/code/task3.py:11:ap.add_argument("--epochs", type=int, default=30)

In [11]:
import json, os, zipfile
r = json.load(open("/content/results.json"))
r["name"] = "Kasak Malhotra"
r["roll_no"] = "24b0418"
r["training_setup"]["augmentation"] = "random crop with padding + horizontal flip"   # edit if you changed it
r["task5"]["val_acc_synthetic_labels__best_method"] = 0.5299999713897705
json.dump(r, open("/content/results.json", "w"), indent=2)

report = """# Report

Q1. About 40.2% of training labels are wrong. The largest flows are between visually similar classes: truck<->automobile (1,247 and 1,126 images, 13.1% of all wrong labels together) and cat<->dog (924 and 834, 9.7%). Deer is often labelled horse (683) or dog (491). These six flips cover 29.3% of wrong labels, so the rest is spread thinly across other pairs. Within a class, 28% of trucks and 25% of automobiles carry the other's label.

Q2. Memorization begins around epoch 38-40. Until then, accuracy on mislabeled training images stays at about 12-16%, so the network is ignoring the wrong labels and learning real class structure. From epoch ~40 it climbs steadily, reaching 0.66 by epoch 60, while train accuracy on the noisy labels reaches 0.97. Clean validation accuracy peaks near epoch 35 (0.79) and falls to 0.68 by epoch 60. Noisy validation also peaks at epoch 35, so early stopping near epoch 35 can be chosen without clean labels.

Q3. I scored training images by 5-fold CV (30 epochs) with loss, margin and disagreement. Disagreement had the best AUROC (0.928; loss 0.927, margin 0.898); precision@k was about 0.82 for all three (margin 0.824). Failure case: of the top 18,109 scored images, 3,294 (18%) were actually clean, mostly hard animal images (cat 511, bird 467, deer 442, dog 406). Missed noise was dominated by systematic look-alike flips, truck->automobile (651) and cat->dog (444), which the model predicts consistently, so they look clean.

Q4. I trained three methods: symmetric cross-entropy (noisy-val accuracy 0.552), small-loss filtering (0.558) and co-teaching (0.571, best). Co-teaching helps because two networks each choose small-loss samples for the other, so neither reinforces its own confirmation errors. The forget rate was chosen on noisy validation accuracy only, never clean labels. All three beat plain noisy CE at epoch 60 (0.496), but only co-teaching beat early stopping (0.559), by about 1.2 points; sce and filter fell slightly below it.

Q5. On the synthetic-label task the baseline reached 0.424 validation accuracy on synthetic labels. The best method (two-network small-loss selection, run f40) reached 0.530, a gain of 10.6 points. The choice was made on synthetic validation labels only; clean labels were never used for selection.
"""
open("/content/report.md", "w").write(report)

def cmd(script, extra=""):
    src = open(f"/content/code/{script}").read()
    flags = "--data /content/data" if '"--data"' in src else ""
    return " ".join(f"python code/{script} {flags} {extra}".split())

readme = f"""# Code README

Setup: Colab T4, seed 0, data in /content/data, run from the folder that contains code/.

## Commands
- Task 1: {cmd('task1.py')}
- Task 2: {cmd('task2.py')}
- Task 3: {cmd('task3.py')}
- Task 4 sce, filter: {cmd('task4.py', '--methods sce,filter')}
- Task 4 coteach: {cmd('task4_coteach.py')}
- Task 5: {cmd('task5.py')}  (submitted best run is tagged f40, see outputs/t5_f40_summary.json)
Shared code: common.py, model.py, train.py. test_common.py is a unit test.

## Submitted vs exploratory
Submitted Task 4 methods: sce, filter, coteach (best: coteach).

## Label usage
Clean labels were used only for measurement. val_acc_clean was logged but never used for model selection, early stopping or hyperparameter choice. The forget rate was chosen on noisy validation accuracy; Task 5 selection used synthetic validation labels.
"""
open("/content/code/README.md", "w").write(readme)
print(readme)

out = "/content/24b0418_cv_bootcamp.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["results.json", "report.md"]:
        z.write(f"/content/{f}", f)
    for dp, dn, fs in os.walk("/content/predictions"):
        for f in fs: z.write(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), "/content"))
    for dp, dn, fs in os.walk("/content/code"):
        dn[:] = [x for x in dn if x != "__pycache__"]
        for f in fs:
            if f.endswith((".npy", ".pt", ".pth", ".pyc")): continue
            z.write(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), "/content"))
print("zip MB:", os.path.getsize(out) / 1e6)

# Code README

Setup: Colab T4, seed 0, data in /content/data, run from the folder that contains code/.

## Commands
- Task 1: python code/task1.py --data /content/data
- Task 2: python code/task2.py
- Task 3: python code/task3.py --data /content/data
- Task 4 sce, filter: python code/task4.py --data /content/data --methods sce,filter
- Task 4 coteach: python code/task4_coteach.py --data /content/data
- Task 5: python code/task5.py --data /content/data  (submitted best run is tagged f40, see outputs/t5_f40_summary.json)
Shared code: common.py, model.py, train.py. test_common.py is a unit test.

## Submitted vs exploratory
Submitted Task 4 methods: sce, filter, coteach (best: coteach).
Exploratory runs also executed and not submitted: fsce (task4_fsce.py), ls, relabel (task4_relabel.py), plus and cotplus (task4_cotplus.py), and extra forget rates f35, f40, f45. Their logs are in outputs/.

## Label usage
Clean labels were used only for measurement. val_acc_clean was logged but never use

In [12]:
!cd /content/cvzip/CV && find . -type f -size -2M | grep -v -E "\.(npy|pt|pth)$"
!cd /content/cvzip/CV && ls -la

./results_template.json
./data/train_idx.txt
./data/val_idx.txt
./data/classes.txt
./data/labels.csv
./model.py
./CV Problem Statement.docx
./check_format.py
total 256
drwxr-xr-x 3 root root   4096 Oct 19 17:31  .
drwxr-xr-x 3 root root   4096 Oct 19 17:26  ..
-rw-r--r-- 1 root root  16063 Oct  5 09:04  check_format.py
-rw-r--r-- 1 root root 221660 Oct  5 09:04 'CV Problem Statement.docx'
drwxr-xr-x 2 root root   4096 Oct 19 17:31  data
-rw-r--r-- 1 root root   2542 Oct  5 09:04  model.py
-rw-r--r-- 1 root root    774 Oct  5 09:04  results_template.json


In [13]:
!pip -q install python-docx
import docx
doc = docx.Document("/content/cvzip/CV/CV Problem Statement.docx")
for p in doc.paragraphs:
    if p.text.strip(): print(p.text)
for t in doc.tables:
    for row in t.rows:
        print(" | ".join(c.text.strip() for c in row.cells))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 7.7 MB/s eta 0:00:00
Inter-IIT Tech Meet 15.0 Selection Bootcamp
Inter-IIT Tech Meet 15.0 Selection Bootcamp - Computer Vision 
Submission: a single file <rollno>_cv_bootcamp.zip containing:
code/          # notebook(s) or scripts; README.md with the exact commands to reproduce each task
predictions/   # files listed under each problem
results.json   # filled-in copy of the template: seed, settings, method names, validation numbers
report.md      # answers to the fixed questions (Q1-Q5), ≤ 100 words each
Problem Statement - Learning from Messy Human Labels (CIFAR-10N)
Real-world labels are noisy, and human annotators make systematic mistakes rather than random ones. CIFAR-10N re-labels the CIFAR-10 training set with real crowd-worker annotations. In the "worst" label set used here, about 40% of training labels are wrong.
Your job is to train a good classifier from noisy labels, find the mislabeled images, and understand how hum

In [14]:
import re, os, zipfile
q5 = """Q5. No, the ranking was the same on both noise types: co-teaching beat filtering, which beat the plain cross-entropy baseline (real: 0.571 / 0.558 / 0.496; synthetic: 0.530 / 0.504 / 0.424). The gap widened on synthetic noise (+10.6 points vs +7.5), and the baseline fell lower. Uniform flips ignore image content, so they show up as high-loss samples and small-loss selection removes them cleanly. Real flips follow look-alike classes (truck/automobile, cat/dog), and models predict them consistently, so they look clean (see Q3).
"""
t = open("/content/report.md").read()
t = re.split(r"\nQ5\.", t)[0].rstrip() + "\n\n" + q5
open("/content/report.md", "w").write(t)
for line in t.splitlines():
    if line.startswith("Q"): print(line[:2], len(line.split()), "words")

out = "/content/24b0418_cv_bootcamp.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["results.json", "report.md"]: z.write(f"/content/{f}", f)
    for dp, dn, fs in os.walk("/content/predictions"):
        for f in fs: z.write(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), "/content"))
    for dp, dn, fs in os.walk("/content/code"):
        dn[:] = [x for x in dn if x != "__pycache__"]
        for f in fs:
            if f.endswith((".npy", ".pt", ".pth", ".pyc")): continue
            z.write(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), "/content"))
!python /content/cvzip/CV/check_format.py /content/24b0418_cv_bootcamp.zip
!cp /content/24b0418_cv_bootcamp.zip /content/report.md /content/results.json /content/drive/MyDrive/cv-backup/

Q1 73 words
Q2 84 words
Q3 81 words
Q4 77 words
Q5 83 words

File check
------------------------------------------------------------
  t1_clean_ref.csv           OK
  t1_noisy_ce.csv            OK
  t2_dynamics.csv            OK
  t2_early_stop.csv          OK
  t3_noise_scores.csv        OK
  t4_coteach.csv             OK
  t4_filter.csv              OK
  t4_sce.csv                 OK
  t5_baseline_synth.csv      OK
  t5_best_synth.csv          OK

Task completeness (1.0 = all files valid)
------------------------------------------------------------
  task1: 1.00
  task2: 1.00
  task3: 1.00
  task4: 1.00
  task5: 1.00

All good.
cp: target '/content/drive/MyDrive/cv-backup/': No such file or directory


In [15]:
import os, zipfile, numpy as np, pandas as pd, torch

# 1) README transparency note
p = "/content/code/README.md"; s = open(p).read()
if "rules allow 1-3 methods" not in s:
    s = s.replace("## Label usage", "Note: the rules allow 1-3 methods; I submitted 3 (sce, filter, coteach) but also ran the exploratory variants above while tuning, and I am reporting them for transparency. Selection used noisy validation accuracy only.\n\n## Label usage")
    open(p, "w").write(s)

# 2) rezip
out = "/content/24b0418_cv_bootcamp.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["results.json", "report.md"]: z.write(f"/content/{f}", f)
    for dp, dn, fs in os.walk("/content/predictions"):
        for f in fs: z.write(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), "/content"))
    for dp, dn, fs in os.walk("/content/code"):
        dn[:] = [x for x in dn if x != "__pycache__"]
        for f in fs:
            if f.endswith((".npy", ".pt", ".pth", ".pyc")): continue
            z.write(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), "/content"))
print(os.path.getsize(out)/1e6, "MB")

# 3) Task 4 vs Task 5 consistency
def ld(f):
    x = torch.load(f, map_location="cpu"); return np.asarray(x.cpu() if hasattr(x, "cpu") else x).ravel()
a = pd.read_csv("/content/predictions/t4_coteach.csv").pred.values
print("submitted t4_coteach == f40 run:", (a == ld("/content/outputs/t4_f40_test_preds.pt")).mean())
print("submitted t4_coteach == coteach run:", (a == ld("/content/outputs/t4_coteach_test_preds.pt")).mean())
print(open("/content/outputs/t5_f40_summary.json").read())
!grep -n -i "forget" /content/code/task4_coteach.py | head -5
!grep -n -i "forget\|drop" /content/code/task5.py | head -5

# 4) check + backup (creates the folder this time)
!python /content/cvzip/CV/check_format.py /content/24b0418_cv_bootcamp.zip | tail -3
!mkdir -p /content/drive/MyDrive/cv-backup
!cp -r /content/24b0418_cv_bootcamp.zip /content/report.md /content/results.json /content/code /content/predictions /content/outputs /content/drive/MyDrive/cv-backup/
!ls /content/drive/MyDrive/cv-backup

0.536022 MB


ValueError: operands could not be broadcast together with shapes (10000,) (600000,) 

In [16]:
import numpy as np, pandas as pd, torch, glob
def ld(f):
    x = torch.load(f, map_location="cpu")
    if isinstance(x, dict): print(f.split("/")[-1], "dict keys:", list(x.keys())); return None
    return np.asarray(x.cpu() if hasattr(x, "cpu") else x)
a = pd.read_csv("/content/predictions/t4_coteach.csv").pred.values
for name in ["t4_f40", "t4_coteach", "t4_f45"]:
    x = ld(f"/content/outputs/{name}_test_preds.pt")
    if x is None: continue
    print(name, "shape", x.shape)
    if x.size % 10000 == 0:
        x = x.reshape(-1, 10000)
        m = [(a == r).mean() for r in x]
        print("  match with last row:", round(m[-1], 4), "| best row:", int(np.argmax(m)), round(max(m), 4))
print(open("/content/outputs/t5_f40_summary.json").read())
!grep -n -i "forget" /content/code/task4_coteach.py | head -5
!grep -n -i "forget\|drop" /content/code/task5.py | head -5

!python /content/cvzip/CV/check_format.py /content/24b0418_cv_bootcamp.zip | tail -3
!mkdir -p /content/drive/MyDrive/cv-backup
!cp -r /content/24b0418_cv_bootcamp.zip /content/report.md /content/results.json /content/code /content/predictions /content/outputs /content/drive/MyDrive/cv-backup/
!ls /content/drive/MyDrive/cv-backup

t4_f40 shape (60, 10000)
  match with last row: 1.0 | best row: 59 1.0
t4_coteach shape (60, 10000)
  match with last row: 0.9084 | best row: 58 0.9105
t4_f45 shape (60, 10000)
  match with last row: 0.9111 | best row: 59 0.9111
{
  "final_val_acc_noisy": 0.5299999713897705,
  "best_val_acc_noisy": 0.5313999652862549,
  "final_val_acc_clean": 0.8903999924659729
}
1:"""Task 4 candidate: co-teaching (2 networks, small-loss exchange). Same recipe as Task 1. Forget rate is a fixed hyperparameter."""
13:ap.add_argument("--forget", type=float, default=0.35)
44:    forget = args.forget * min(1.0, (ep - 1) / args.ramp)
54:        k = max(1, int(round((1 - forget) * len(b))))
63:    row = {"epoch": ep, "train_loss": run_loss / seen, "forget": forget,
15:ap.add_argument("--drop", type=float, default=0.35)
61:keep_np = np.where(score.numpy() <= np.quantile(score.numpy(), 1 - args.drop))[0]
63:dropped = np.setdiff1d(np.arange(N), keep_np)
64:print(f"keeps {len(keep)}; fraction of dropped truly wro

In [17]:
import numpy as np, pandas as pd, torch
a = pd.read_csv("/content/predictions/t4_coteach.csv").pred.values
for name in ["t4_f40", "t4_coteach"]:
    x = torch.load(f"/content/outputs/{name}_test_preds.pt", map_location="cpu")
    x = np.asarray(x.cpu() if hasattr(x, "cpu") else x).reshape(-1, 10000)
    print(name, "match with last row:", (a == x[-1]).mean())

t4_f40 match with last row: 1.0
t4_coteach match with last row: 0.9084


In [18]:
import json
p = "/content/results.json"
r = json.load(open(p))
for m in r["task4"]["methods"]:
    if m["name"] == "coteach":
        m["val_acc_noisy_labels"] = 0.5758
json.dump(r, open(p, "w"), indent=2)
print([ (m["name"], m["val_acc_noisy_labels"]) for m in r["task4"]["methods"] ])

[('sce', 0.5515999794006348), ('filter', 0.5577999949455261), ('coteach', 0.5758)]


In [19]:
import os, zipfile
out = "/content/24b0418_cv_bootcamp.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["results.json", "report.md"]: z.write(f"/content/{f}", f)
    for dp, dn, fs in os.walk("/content/predictions"):
        for f in fs: z.write(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), "/content"))
    for dp, dn, fs in os.walk("/content/code"):
        dn[:] = [x for x in dn if x != "__pycache__"]
        for f in fs:
            if f.endswith((".npy", ".pt", ".pth", ".pyc")): continue
            z.write(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), "/content"))
print(os.path.getsize(out)/1e6, "MB")
!python /content/cvzip/CV/check_format.py /content/24b0418_cv_bootcamp.zip | tail -3
!mkdir -p /content/drive/MyDrive/cv-backup
!cp -r /content/24b0418_cv_bootcamp.zip /content/report.md /content/results.json /content/code /content/predictions /content/outputs /content/drive/MyDrive/cv-backup/

0.536015 MB
  task5: 1.00

All good.


In [20]:
import zipfile, os

zip_path = "/content/24b0418_cv_bootcamp.zip"

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        files = z.namelist()
        print("ZIP contents:")
        for f in files:
            print(f)

        print("\nNotebook files:")
        notebooks = [f for f in files if f.endswith(".ipynb")]
        print(notebooks if notebooks else "No .ipynb notebook found inside ZIP")
else:
    print("ZIP not found at", zip_path)

ZIP contents:
results.json
report.md
predictions/t1_noisy_ce.csv
predictions/t2_early_stop.csv
predictions/t2_dynamics.csv
predictions/t3_noise_scores.csv
predictions/t4_coteach.csv
predictions/t4_sce.csv
predictions/t5_baseline_synth.csv
predictions/t4_filter.csv
predictions/t1_clean_ref.csv
predictions/t5_best_synth.csv
code/task4_fsce.py
code/task2.py
code/model.py
code/task4_relabel.py
code/test_common.py
code/train.py
code/common.py
code/task4_coteach.py
code/task4.py
code/README.md
code/task5.py
code/task4_cotplus.py
code/task1.py
code/task3.py

Notebook files:
No .ipynb notebook found inside ZIP


In [21]:
import json

with open("/content/results.json") as f:
    r = json.load(f)

print(json.dumps(r["task4"], indent=2))

{
  "methods": [
    {
      "name": "sce",
      "file": "t4_sce.csv",
      "val_acc_noisy_labels": 0.5515999794006348
    },
    {
      "name": "filter",
      "file": "t4_filter.csv",
      "val_acc_noisy_labels": 0.5577999949455261
    },
    {
      "name": "coteach",
      "file": "t4_coteach.csv",
      "val_acc_noisy_labels": 0.5758
    }
  ],
  "best_method": "coteach"
}
